<a href="https://colab.research.google.com/github/imtiaz-techie/multimodal-ddi-checker/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

df_med = pd.read_csv("/medicine.csv")
df_gen = pd.read_csv("/generic.csv")

print("=== MEDICINE.CSV COLUMNS ===")
print(df_med.columns.tolist())

print("\n=== GENERIC.CSV COLUMNS ===")
print(df_gen.columns.tolist())

print("\n=== SAMPLE ROW FROM GENERIC.CSV ===")
for col in df_gen.columns:
    print(f"\n[{col}]:\n{df_gen[col].iloc[0]}")

=== MEDICINE.CSV COLUMNS ===
['brand id', 'brand name', 'type', 'slug', 'dosage form', 'generic', 'strength', 'manufacturer', 'package container', 'Package Size']

=== GENERIC.CSV COLUMNS ===
['generic id', 'generic name', 'slug', 'monograph link', 'drug class', 'indication', 'indication description', 'therapeutic class description', 'pharmacology description', 'dosage description', 'administration description', 'interaction description', 'contraindications description', 'side effects description', 'pregnancy and lactation description', 'precautions description', 'pediatric usage description', 'overdose effects description', 'duration of treatment description', 'reconstitution description', 'storage conditions description', 'descriptions count']

=== SAMPLE ROW FROM GENERIC.CSV ===

[generic id]:
31

[generic name]:
Adalimumab

[slug]:
adalimumab-31

[monograph link]:
https://medex.com.bd/attachments/FgGWUFrlD7LThDOJE9ArMGq3BaZ3J3/adalimumab-prescribing-information

[drug class]:
Immun

In [4]:
from bs4 import BeautifulSoup
import pandas as pd

# (Porobortite notun session-e shudhu nicher 2 line uncomment kore GitHub theke direct load korte parbe)
# !git clone https://github.com/TOMAR_GITHUB_USERNAME/multimodal-ddi-checker.git
# df_med = pd.read_csv("multimodal-ddi-checker/medicine.csv")
# df_gen = pd.read_csv("multimodal-ddi-checker/generic.csv")

# 1. generic.csv-er shobgulo column-er nam niche niche dekha
print("=== ALL COLUMNS IN GENERIC.CSV ===")
for i, col in enumerate(df_gen.columns):
    print(f"{i+1}. {col}")

# 2. HTML tag clean korar function
def clean_html(raw_html):
    if pd.isna(raw_html):
        return "No information available."
    return BeautifulSoup(str(raw_html), "html.parser").get_text(separator=" ", strip=True)

# 3. Brand Name theke Generic ebong Clinical Info ber korar function
def lookup_medicine_details(brand_name):
    med_match = df_med[df_med['brand name'].str.lower().str.strip() == brand_name.lower().strip()]
    if med_match.empty:
        print(f"\nBrand '{brand_name}' database-e pawa jayni.")
        return

    generic_raw = med_match.iloc[0]['generic']
    strength = med_match.iloc[0]['strength']
    dosage = med_match.iloc[0]['dosage form']

    print(f"\n==========================================")
    print(f"Brand Name : {med_match.iloc[0]['brand name']} ({dosage}, {strength})")
    print(f"Generic    : {generic_raw}")

    # Combination drug ('+' thakle) alada kore check kora
    generics = [g.strip() for g in str(generic_raw).split('+')]

    for gen in generics:
        gen_match = df_gen[df_gen['generic name'].str.lower().str.strip() == gen.lower()]
        if not gen_match.empty:
            row = gen_match.iloc[0]
            print(f"\n--- Clinical Data for: {gen} ---")
            for col in df_gen.columns:
                if any(k in col.lower() for k in ['interaction', 'precaution', 'contraindication', 'side_effect', 'side effect']):
                    print(f"[{col}]: {clean_html(row[col])[:250]}...")
        else:
            print(f"\n[!] '{gen}'-er exact match generic.csv-te pawa jayni (Salt cleaning dorkar).")

# 4. Popular Bangladeshi oushudh diye test kora
for test_brand in ["Napa", "Seclo", "Ecoprin", "A-Cof"]:
    lookup_medicine_details(test_brand)

=== ALL COLUMNS IN GENERIC.CSV ===
1. generic id
2. generic name
3. slug
4. monograph link
5. drug class
6. indication
7. indication description
8. therapeutic class description
9. pharmacology description
10. dosage description
11. administration description
12. interaction description
13. contraindications description
14. side effects description
15. pregnancy and lactation description
16. precautions description
17. pediatric usage description
18. overdose effects description
19. duration of treatment description
20. reconstitution description
21. storage conditions description
22. descriptions count

Brand Name : Napa (Tablet, 500 mg)
Generic    : Paracetamol

--- Clinical Data for: Paracetamol ---
[interaction description]: Patients who have taken barbiturates, tricyclic antidepressants and alcohol may show diminished ability to metabolise large doses of Paracetamol. Alcohol can increase the hepatotoxicity of Paracetamol overdosage. Chronic ingestion of anticonvulsants ...
[contra

In [5]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def clean_html(raw_html):
    if pd.isna(raw_html):
        return "No information available."
    return BeautifulSoup(str(raw_html), "html.parser").get_text(separator=" ", strip=True)

# 1. Generic.csv-te Smart Matching Function (Combination + Salt Handled)
def find_generic_monographs(generic_raw):
    matched_rows = []
    gen_clean = str(generic_raw).strip()

    # Step A: Age puro নামটা (Combination shoho) direct ache kina check kora
    exact_combo = df_gen[df_gen['generic name'].str.lower().str.strip() == gen_clean.lower()]
    if not exact_combo.empty:
        matched_rows.append(exact_combo.iloc[0])
        return matched_rows

    # Step B: Direct na pele '+' diye bhenge protiti upadan alada kore khoja
    parts = [p.strip() for p in gen_clean.split('+')]
    for part in parts:
        # Exact match khoja
        m = df_gen[df_gen['generic name'].str.lower().str.strip() == part.lower()]
        if m.empty:
            # Salt ba bracket bad diye prothom mul shobdo (base chemical) diye khoja
            base_word = re.sub(r'\(.*?\)', '', part).strip().split()[0].lower()
            # Shudhu single generic row-gulote khoja (jekhane '+' nei)
            single_gens = df_gen[~df_gen['generic name'].str.contains(r'\+', na=False)]
            m = single_gens[single_gens['generic name'].str.lower().str.startswith(base_word)]

        if not m.empty:
            matched_rows.append(m.iloc[0])

    return matched_rows

# 2. Smart Brand Lookup Function
def smart_lookup_medicine(brand_query):
    q = brand_query.lower().strip()
    # Age exact match, na pele startswith diye khoja
    med_match = df_med[df_med['brand name'].str.lower().str.strip() == q]
    if med_match.empty:
        med_match = df_med[df_med['brand name'].str.lower().str.startswith(q)]

    if med_match.empty:
        print(f"\n[!] Brand '{brand_query}' database-e pawa jayni.")
        return

    # Tablet ba Capsule age priority dewa (Injection-er bodole)
    oral_match = med_match[med_match['dosage form'].str.contains('Tablet|Capsule|Syrup', case=False, na=False)]
    row_med = oral_match.iloc[0] if not oral_match.empty else med_match.iloc[0]

    generic_raw = row_med['generic']
    print(f"\n==========================================")
    print(f"Brand Name : {row_med['brand name']} ({row_med['dosage form']}, {row_med['strength']})")
    print(f"Generic    : {generic_raw}")

    monographs = find_generic_monographs(generic_raw)
    if not monographs:
        print(f"[!] '{generic_raw}'-er kono monograph generic.csv-te pawa jayni.")
        return

    for row in monographs:
        print(f"\n--- Clinical Data for: {row['generic name']} ---")
        for col in ['drug class', 'interaction description', 'contraindications description', 'precautions description']:
            if col in df_gen.columns:
                print(f"[{col}]: {clean_html(row[col])[:250]}...")

# 3. Abar Test kora (Seclo, A-Cof, Sergel, Monas)
for test_brand in ["Seclo", "A-Cof", "Sergel", "Monas"]:
    smart_lookup_medicine(test_brand)


Brand Name : Seclo (Capsule (Delayed Release), 40 mg)
Generic    : Omeprazole

--- Clinical Data for: Omeprazole ---
[drug class]: Proton Pump Inhibitor...
[interaction description]: Omeprazole can prolong the elimination of diazepam, warfarin and phenytoin. So, reduction of warfarin or phenytoin dose may be necessary when Omeprazole is added to the treatment. There is no evidence of an interaction of Omeprazole with theophylline...
[contraindications description]: Omeprazole is contraindicated in patients with known hypersensitivity to any of the components of the formulation....
[precautions description]: When gastric ulcer is suspected, the possibility of gastric malignancy should be excluded before treatment with Omeprazole is instituted, as treatment may alleviate the symptoms and delay diagnosis....

Brand Name : A-Cof (Syrup, (10 mg+30 mg+1.25 mg)/5 ml)
Generic    : Dextromethorphan + Pseudoephedrine + Triprolidine

--- Clinical Data for: Dextromethorphan + Pseudoephedrine + Tr

In [10]:
!pip install -q -U google-genai

from google.colab import userdata
from google import genai

# ১. Secrets থেকে Key নিয়ে অটোমেটিক স্পেস বা কোটেশন পরিষ্কার করা
raw_key = userdata.get('GEMINI_API_KEY')
GEMINI_API_KEY = raw_key.strip().replace('"', '').replace("'", "")

print(f"Key loaded successfully! (Starts with: {GEMINI_API_KEY[:6]}..., Length: {len(GEMINI_API_KEY)})")
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Prescription-er shob brand theke grounded clinical context toiri kora
def build_grounded_context(brand_list):
    context_list = []
    mapped_brands = []

    for brand_query in brand_list:
        q = brand_query.lower().strip()
        med_match = df_med[df_med['brand name'].str.lower().str.strip() == q]
        if med_match.empty:
            med_match = df_med[df_med['brand name'].str.lower().str.startswith(q)]

        if med_match.empty:
            mapped_brands.append(f"- {brand_query}: Database-e pawa jayni")
            continue

        oral_match = med_match[med_match['dosage form'].str.contains('Tablet|Capsule|Syrup', case=False, na=False)]
        row_med = oral_match.iloc[0] if not oral_match.empty else med_match.iloc[0]

        b_name = row_med['brand name']
        dosage = row_med['dosage form']
        strength = row_med['strength']
        generic_raw = row_med['generic']

        mapped_brands.append(f"- {b_name} ({dosage}, {strength}) -> Generic: {generic_raw}")

        # Ager cell-er find_generic_monographs function use kore data ana
        monographs = find_generic_monographs(generic_raw)
        for row in monographs:
            info = (
                f"Brand: {b_name} | Generic: {row['generic name']}\n"
                f"Drug Class: {clean_html(row.get('drug class', ''))}\n"
                f"Interactions: {clean_html(row.get('interaction description', ''))}\n"
                f"Contraindications: {clean_html(row.get('contraindications description', ''))}\n"
                f"Precautions: {clean_html(row.get('precautions description', ''))}"
            )
            context_list.append(info)

    return "\n".join(mapped_brands), "\n\n---\n\n".join(context_list)

 # 3. Gemini RAG Pipeline diye Interaction ও Duplicate Therapy Check kora
def check_prescription_safety(brand_list):
    summary, context_data = build_grounded_context(brand_list)
    print("=== PRESCRIPTION MAPPING ===")
    print(summary)
    print("\nGenerating Grounded Clinical DDI Report...\n")

    prompt = f"""
    Nicher prescription-er oushudh-gulor moddhe kono Drug-Drug Interaction (DDI), Duplicate Therapy (ekoi group-er duita oushudh), ba jhuki ache kina ta shudhumatro nicher 'Clinical Database Context' bebohar kore vishleshon koro:

    Prescribed Medicines:
    {summary}

    Clinical Database Context:
    {context_data}

    Output-e ৩ti section rakhbe:
    1. Drug-Drug Interaction & Duplicate Therapy Alert (jemon: Omeprazole o Esomeprazole eksathe thakle ba Omeprazole-er sathe Diazepam/Warfarin thakle warning দাও)
    2. Contraindications & Precautions Summary
    3. Final Recommendation & Medical Disclaimer
    """

    # Account-e thaka shob Flash model khuje ber kora
    flash_models = [
        m.name for m in client.models.list()
        if "gemini" in m.name.lower() and "flash" in m.name.lower()
        and not any(x in m.name.lower() for x in ["tts", "image", "audio", "live"])
    ]
    print("Available Flash Models in your account:", flash_models)

    # Eke eke model try kore jeta সচল ache sheta diye report generate kora
    for model_name in flash_models:
        try:
            print(f"Trying model: {model_name} ...")
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
            )
            print(f"✅ Successfully generated with: {model_name}\n" + "="*50 + "\n")
            return response.text
        except Exception as e:
            print(f"⚠️ Skipped {model_name}")

    return "Kono working model pawa jayni."

# 4. Test Run: Seclo (Omeprazole), Sergel (Esomeprazole), Sedil (Diazepam), ebong A-Cof
test_prescription = ["Seclo", "Sergel", "Sedil", "A-Cof"]
print(check_prescription_safety(test_prescription))

Key loaded successfully! (Starts with: AQ.Ab8..., Length: 53)
=== PRESCRIPTION MAPPING ===
- Seclo (Capsule (Delayed Release), 40 mg) -> Generic: Omeprazole
- Sergel (Tablet (Enteric Coated), 40 mg) -> Generic: Esomeprazole
- Sedil (Tablet, 5 mg) -> Generic: Diazepam
- A-Cof (Syrup, (10 mg+30 mg+1.25 mg)/5 ml) -> Generic: Dextromethorphan + Pseudoephedrine + Triprolidine

Generating Grounded Clinical DDI Report...

Available Flash Models in your account: ['models/gemini-2.5-flash', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash']
Trying model: models/gemini-2.5-flash ...
⚠️ Skipped models/gemini-2.5-flash
Trying model: models/gemin